In [4]:
%load_ext autoreload
%autoreload 2
    
import os
import sys
import pandas as pd
import xarray as xr
import numpy as np
from dask.distributed import Client
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
sys.path.append('/home/563/sc1326/repos/cdrmip_extremes')
from cdrmip_extremes.configs import data_dir, models, expts
from cdrmip_extremes.load_data import load_raw_tas, load_tas_anom
from cdrmip_extremes.utils import global_mean, calc_anomaly, concat_branches
from cdrmip_extremes.sat import find_crossing_years, find_matching_gwls
from cdrmip_extremes.CMIP6_file_search import find_CMIP_files, search_and_load

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
client = Client(memory_limit=None,threads_per_worker=1,n_workers=28)

# Load ssp534-over and ssp126 extension tas data

In [8]:
expts = ['ssp534-over','ssp126']
chunks = {'time':365}
vars_to_drop=['height','time_bnds','lat_bnds','lon_bnds','areacella']
table_id = 'Amon'
variable_id = 'tas'

In [ ]:
data = {}
for experiment_id in expts:
    data[experiment_id] = search_and_load(
        experiment_id, table_id, variable_id, vars_to_drop, chunks,
        threshold=2102, check_end_year=True
    )

Searching for ssp534-over files
Loading ACCESS-CM2 r1i1p1f1
ACCESS-CM2 r1i1p1f1: 2041-01-16 12:00:00 → 2100-12-16 12:00:00
  Skipping — end year 2100 not beyond 2102
Loading ACCESS-ESM1-5 r10i1p1f2
ACCESS-ESM1-5 r10i1p1f2: 2041-01-16 12:00:00 → 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 r11i1p1f2
ACCESS-ESM1-5 r11i1p1f2: 2041-01-16 12:00:00 → 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 r12i1p1f2
ACCESS-ESM1-5 r12i1p1f2: 2041-01-16 12:00:00 → 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 r13i1p1f2
ACCESS-ESM1-5 r13i1p1f2: 2041-01-16 12:00:00 → 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 r14i1p1f2
ACCESS-ESM1-5 r14i1p1f2: 2041-01-16 12:00:00 → 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 r15i1p1f2
ACCESS-ESM1-5 r15i1p1f2: 2041-01-16 12:00:00 → 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 r16i1p1f2
ACCESS-ESM1-5 r16i1p1f2: 2041-01-16 12:00:00 → 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 r17i1p1f2
ACCESS-ESM1-5 r17i1p1f2: 2041-01-16 12:00:00 → 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 r18i1p1f2
ACCESS-ESM

## Concatenate

In [ ]:
# Step 1: chain ssp534-over -> ssp585
for model, member_dict in data['ssp534-over'].items():
    for member, child_ds in member_dict.items():
        parent_ds = load_scenariomip_data.search_and_load_parent(child_ds,chunks=chunks,vars_to_drop=vars_to_drop)
        print(f"Concatenating ssp534-over to {parent_ds.attrs['experiment_id']} [{model}/{member}]")
        if model == 'ACCESS-ESM1-5':
            branch_date = '2041-01-01 00:00:00'
        else:
            branch_date=None
        data['ssp534-over'][model][member] = load_scenariomip_data.concatenate_branches(
            child_ds, parent_ds, time_dim='time',
            branch_date=branch_date
        )

# Step 2: chain all experiments -> historical
concat = defaultdict(lambda: defaultdict(dict))
for expt, model_dict in data.items():
    for model, member_dict in model_dict.items():
        for member, child_ds in member_dict.items():
            parent_ds = load_scenariomip_data.search_and_load_parent(child_ds,chunks=chunks,vars_to_drop=vars_to_drop)
            print(f"Concatenating {expt} to {parent_ds.attrs['experiment_id']} [{model}/{member}]")
            branch_date = '2015-01-01'
            concat[model][expt][member] = load_scenariomip_data.concatenate_branches(
                child_ds, parent_ds, time_dim='time',
                branch_date=branch_date,
            )

## Create DataTree

In [9]:
dt = xr.DataTree.from_dict({
    f"/{model}/{expt}/{member}": ds
    for model, expt_dict in concat.items()
    for expt, member_dict in expt_dict.items()
    for member, ds in member_dict.items()
})
dt

<xarray.DataTree>
Group: /
├── Group: /ACCESS-ESM1-5
│   ├── Group: /ACCESS-ESM1-5/ssp534-over
│   │   ├── Group: /ACCESS-ESM1-5/ssp534-over/r10i1p1f2
│   │   │       Dimensions:  (time: 5412, lat: 145, lon: 192)
│   │   │       Coordinates:
│   │   │         * time     (time) object 43kB 1850-01-16 12:00:00 ... 2300-12-16 12:00:00
│   │   │         * lat      (lat) float64 1kB -90.0 -88.75 -87.5 -86.25 ... 87.5 88.75 90.0
│   │   │         * lon      (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 352.5 354.4 356.2 358.1
│   │   │       Data variables:
│   │   │           tas      (time, lat, lon) float32 603MB dask.array<chunksize=(365, 145, 192), meta=np.ndarray>
│   │   │       Attributes: (12/47)
│   │   │           Conventions:            CF-1.7 CMIP-6.2
│   │   │           activity_id:            CMIP
│   │   │           branch_method:          standard
│   │   │           branch_time_in_child:   0.0
│   │   │           branch_time_in_parent:  87658.0
│   │   │           creation_date:          2020-06-05T04:06:11Z
│   │   │           ...                     ...
│   │   │           variable_id:            tas
│   │   │           variant_label:          r10i1p1f1
│   │   │           version:                v20200605
│   │   │           license:                CMIP6 model data produced by CSIRO is licensed un...
│   │   │           cmor_version:           3.4.0
│   │   │           tracking_id:            hdl:21.14100/af78ae5e-f3a6-4e99-8cfe-5f29eb467cd1
│   │   ├── Group: /ACCESS-ESM1-5/ssp534-over/r11i1p1f2
│   │   │       Dimensions:  (time: 5412, lat: 145, lon: 192)
│   │   │       Coordinates:
│   │   │         * time     (time) object 43kB 1850-01-16 12:00:00 ... 2300-12-16 12:00:00
│   │   │         * lat      (lat) float64 1kB -90.0 -88.75 -87.5 -86.25 ... 87.5 88.75 90.0
│   │   │         * lon      (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 352.5 354.4 356.2 358.1
│   │   │       Data variables:
│   │   │           tas      (time, lat, lon) float32 603MB dask.array<chunksize=(365, 145, 192), meta=np.ndarray>
│   │   │       Attributes: (12/47)
│   │   │           Conventions:            CF-1.7 CMIP-6.2
│   │   │           activity_id:            CMIP
│   │   │           branch_method:          standard
│   │   │           branch_time_in_child:   0.0
│   │   │           branch_time_in_parent:  94963.0
│   │   │           creation_date:          2020-08-03T02:59:16Z
│   │   │           ...                     ...
│   │   │           variable_id:            tas
│   │   │           variant_label:          r11i1p1f1
│   │   │           version:                v20200803
│   │   │           license:                CMIP6 model data produced by CSIRO is licensed un...
│   │   │           cmor_version:           3.4.0
│   │   │           tracking_id:            hdl:21.14100/554df8e8-68db-49db-a476-1fda9e2a55a3
│   │   ├── Group: /ACCESS-ESM1-5/ssp534-over/r12i1p1f2
│   │   │       Dimensions:  (time: 5412, lat: 145, lon: 192)
│   │   │       Coordinates:
│   │   │         * time     (time) object 43kB 1850-01-16 12:00:00 ... 2300-12-16 12:00:00
│   │   │         * lat      (lat) float64 1kB -90.0 -88.75 -87.5 -86.25 ... 87.5 88.75 90.0
│   │   │         * lon      (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 352.5 354.4 356.2 358.1
│   │   │       Data variables:
│   │   │           tas      (time, lat, lon) float32 603MB dask.array<chunksize=(365, 145, 192), meta=np.ndarray>
│   │   │       Attributes: (12/47)
│   │   │           Conventions:            CF-1.7 CMIP-6.2
│   │   │           activity_id:            CMIP
│   │   │           branch_method:          standard
│   │   │           branch_time_in_child:   0.0
│   │   │           branch_time_in_parent:  102268.0
│   │   │           creation_date:          2020-08-03T02:59:09Z
│   │   │           ...                     ...
│   │   │           variable_id:            tas
│   │   │           variant_label:          r12i1p1f1
│   │   │           version:                v20200

## Regrid to common grid

In [10]:
def regrid_ds(ds):
    if 'tas' in ds:
        return processing.regrid_to_common(ds)
    return ds

regridded_dt = dt.map_over_datasets(regrid_ds)

Building regridder for grid (145,) x (192,)
Building regridder for grid (192,) x (288,)
Building regridder for grid (128,) x (256,)
Building regridder for grid (64,) x (128,)
Building regridder for grid (256,) x (512,)
Building regridder for grid (258,) x (512,)
Building regridder for grid (90,) x (144,)
Building regridder for grid (143,) x (144,)
Building regridder for grid (64,) x (128,)
Building regridder for grid (160,) x (320,)
Building regridder for grid (144,) x (192,)


## Calculate Anomalies

In [11]:
reference_period = slice(1850,1900)
def anomaly_func(ds: xr.Dataset) -> xr.Dataset:
    if 'tas' not in ds:
        return ds
    # Annual means — collapses 'time' dim into 'year'
    ds_annual = ds.groupby('time.year').mean(dim='time')
    return utils.calc_anomaly(ds_annual, ds_annual, reference_period, time_dim='year')

anomaly_dt = regridded_dt.map_over_datasets(anomaly_func)

# Calculate global mean

In [14]:
def global_mean(ds: xr.Dataset) -> xr.Dataset:
    if 'tas' not in ds:
        return ds
    return utils.global_mean(ds).to_dataset(name='gsat')



gsat_dt = anomaly_dt.map_over_datasets(global_mean)

## Save Data